<a href="https://colab.research.google.com/github/JonasFanZ/114-2-Programing-Language/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1raqHTcfTEmii7em7_tr0TIdL9EhD8jaCQ2TAICWEHMo/edit?usp=sharing

In [152]:
import gradio as gr
import pandas as pd
import datetime
import gspread
from google.colab import auth
from google.auth import default

In [153]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1raqHTcfTEmii7em7_tr0TIdL9EhD8jaCQ2TAICWEHMo/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "分類", "品項", "金額", "付款人", "備註"]

_auth_done = False
_gc = None
_ws = None

In [154]:
# 顯式重置 Gspread 全域變數以確保重新認證和連線
global _auth_done, _gc, _ws
_auth_done = False
_gc = None
_ws = None
print("Gspread 全域變數已重置。請重新執行儲存格 8fa42c04 和 f3e5fd1b。")

Gspread 全域變數已重置。請重新執行儲存格 8fa42c04 和 f3e5fd1b。


In [155]:
# This cell previously contained a redundant Gradio application. It has been cleared to avoid confusion.
# All primary Gradio application logic and Google Sheet interaction functions are now centralized in cells 8fa42c04 and f3e5fd1b.

In [156]:
# 重新執行導入 GSpread 相關套件的程式碼
import gspread
from google.colab import auth
from google.auth import default

print("套件導入完成。")

套件導入完成。


In [157]:
# 重新執行定義 Sheet URL 和 REQUIRED_COLUMNS 的程式碼
SHEET_URL = "https://docs.google.com/spreadsheets/d/1raqHTcfTEmii7em7_tr0TIdL9EhD8jaCQ2TAICWEHMo/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

# 確保 REQUIRED_COLUMNS 僅包含 5 個預期欄位
REQUIRED_COLUMNS = ["日期", "分類", "品項", "金額", "付款人"]

_auth_done = False
_gc = None
_ws = None

print("Sheet URL 和欄位定義已更新。")

Sheet URL 和欄位定義已更新。


In [160]:
import gspread.exceptions
import pandas as pd
import datetime
from google.colab import auth
from google.auth import default

# 核心函數定義
def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    try:
        if not _auth_done:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            _auth_done = True
        if _ws is None:
            gs = _gc.open_by_url(SHEET_URL)
            try:
                _ws = gs.worksheet(WORKSHEET_NAME)
            except gspread.exceptions.WorksheetNotFound:
                _ws = gs.add_worksheet(title=WORKSHEET_NAME, rows="100", cols="10")
            _ensure_headers()
        return _ws
    except Exception as e:
        print(f"Error: {e}")
        return None

def _ensure_headers():
    rows = _ws.get_all_values()
    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
    elif rows[0] != REQUIRED_COLUMNS:
        _ws.update('1:1', [REQUIRED_COLUMNS])

def _read_df():
    ws = _ensure_auth_and_ws()
    if ws is None: return pd.DataFrame(columns=REQUIRED_COLUMNS)
    values = ws.get_all_values()
    if not values or len(values) < 2: return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0)
    return df

def add_record_logic(date, amt, cat, item, payer, remark):
    try:
        ws = _ensure_auth_and_ws()
        if ws:
            ws.append_row([date, cat, item, int(amt), payer, remark], value_input_option="USER_ENTERED")
            df = _read_df()
            total_out = df["金額"].sum()
            # 這裡簡化邏輯：假設目前沒收入統計，結餘與支出連動
            return gr.update(visible=False), df, f"# 0", f"# {int(total_out)}", f"# {-int(total_out)}"
        return gr.update(visible=True), _read_df(), "# Error", "# Error", "# Error"
    except Exception as e:
        print(f"Add Error: {e}")
        return gr.update(visible=True), _read_df(), "# Error", "# Error", "# Error"

def init_load():
    df = _read_df()
    total_out = df["金額"].sum() if not df.empty else 0
    return df, "# 0", f"# {int(total_out)}", f"# {-int(total_out)}"

In [167]:
import gradio as gr
import datetime
import pandas as pd

# 設定今天的模擬日期
SIMULATED_DATE = "2026-03-12"

def get_monthly_total(month_str):
    df = _read_df()
    if df.empty or "日期" not in df.columns or "金額" not in df.columns:
        return "# $ 0"
    try:
        # 確保日期格式正確並篩選月份
        df['日期'] = pd.to_datetime(df['日期'], errors='coerce')
        mask = df['日期'].dt.strftime('%Y-%m') == month_str
        total = df.loc[mask, '金額'].sum()
        return f"# $ {int(total)}"
    except:
        return "# $ 0"

custom_css = """
.gradio-container {
    max-width: 600px !important;
    margin: 40px auto !important;
    background-color: #ffffff !important;
    border-radius: 24px !important;
    box-shadow: 0 20px 50px rgba(0, 0, 0, 0.1) !important;
    border: 1px solid #eee !important;
    position: relative !important;
    min-height: 700px !important;
    display: flex !important;
    flex-direction: column !important;
    overflow: hidden !important; /* 移除外層捲軸 */
}
.minimal-box { border: none !important; box-shadow: none !important; background: transparent !important; }
.kpi-card { text-align: center; padding: 20px; }
.kpi-main h1 { font-size: 3.2rem !important; margin: 0; color: #2d3436; font-weight: 800 !important; }
.title-area { padding: 20px 0 10px 0; text-align: center; }

/* 紀錄列表區域控制：僅保留單層垂直滑桿 */
.table-container {
    overflow-y: auto !important;
    overflow-x: hidden !important;
    max-height: 500px !important;
    padding-bottom: 100px !important; /* 加強防擋間距 */
    scrollbar-width: thin;
}

/* 強制隱藏 Dataframe 的水平捲軸 */
.table-style table { width: 100% !important; }
.table-style .table-wrap { overflow-x: hidden !important; overflow-y: visible !important; }

/* 右下角固定懸浮按鈕 (FAB) */
.fab-btn {
    position: absolute !important;
    bottom: 30px !important;
    right: 30px !important;
    width: 60px !important;
    height: 60px !important;
    border-radius: 50% !important;
    background: linear-gradient(135deg, #6c5ce7, #a29bfe) !important;
    color: white !important;
    font-size: 30px !important;
    font-weight: bold !important;
    box-shadow: 0 10px 20px rgba(108, 92, 231, 0.4) !important;
    cursor: pointer !important;
    z-index: 1000 !important;
    border: none !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
}

.modal-style {
    position: absolute !important;
    top: 50% !important;
    left: 50% !important;
    transform: translate(-50%, -50%) !important;
    z-index: 2000 !important;
    width: 85% !important;
    backdrop-filter: blur(12px) !important;
    background: rgba(255, 255, 255, 0.9) !important;
    border-radius: 20px !important;
    padding: 25px !important;
    box-shadow: 0 15px 35px rgba(0, 0, 0, 0.2) !important;
    border: 1px solid rgba(255, 255, 255, 0.4) !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    with gr.Column(elem_classes="minimal-box"):
        with gr.Group(elem_classes="title-area"):
            gr.Markdown(f"## 財務概覽")
            month_selector = gr.Dropdown(
                choices=["2026-01", "2026-02", "2026-03", "2026-04"],
                value="2026-03",
                label="選擇月份",
                container=False
            )

        with gr.Column(elem_classes="kpi-card kpi-main"):
            gr.Markdown("當月總支出")
            expense_display = gr.Markdown("# $ 0")

        with gr.Column(elem_classes="table-container"):
            data_table = gr.Dataframe(
                headers=REQUIRED_COLUMNS + ["備註"],
                interactive=False,
                elem_classes="table-style"
            )

        open_modal_btn = gr.Button("+", elem_classes="fab-btn")

    with gr.Group(visible=False, elem_classes="modal-style") as modal_window:
        gr.Markdown("### 📝 新增紀錄")
        with gr.Row():
            date_in = gr.Textbox(label="日期", value=SIMULATED_DATE)
            amt_in = gr.Number(label="金額", value=0, precision=0)
        with gr.Row():
            cat_in = gr.Dropdown(label="分類", choices=["飲食", "交通", "娛樂", "教育", "投資", "模型/遊戲", "其他"], value="飲食")
            item_in = gr.Textbox(label="品項", placeholder="輸入名稱")
        with gr.Row():
            payer_in = gr.Textbox(label="付款人", value="自己")
            remark_in = gr.Textbox(label="備註", placeholder="選填")

        with gr.Row():
            cancel_btn = gr.Button("取消")
            save_btn = gr.Button("✅ 儲存", variant="primary")

    open_modal_btn.click(fn=lambda: gr.update(visible=True), outputs=modal_window)
    cancel_btn.click(fn=lambda: gr.update(visible=False), outputs=modal_window)

    month_selector.change(fn=get_monthly_total, inputs=month_selector, outputs=expense_display)

    save_btn.click(
        fn=add_record_logic,
        inputs=[date_in, amt_in, cat_in, item_in, payer_in, remark_in],
        outputs=[modal_window, data_table, gr.State(), expense_display, gr.State()]
    )

    def initial_load_wrapper():
        df, inc, exp, bal = init_load()
        exp_val = get_monthly_total("2026-03")
        return df, exp_val

    demo.load(fn=initial_load_wrapper, outputs=[data_table, expense_display])

demo.launch(share=True, inline=True)

/tmp/ipykernel_182/1541887496.py:90: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33ece9dc3527031139.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
